This notebook is a basic tutorial that demonstrates how to configure a simulation using Concordia.

<a href="https://colab.research.google.com/github/google-deepmind/concordia/blob/main/examples/tutorial.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# @title Colab-specific setup (use a CodeSpace to avoid the need for this).
try:
  %env COLAB_RELEASE_TAG
except:
  pass  # Not running in colab.
else:
  %pip install --ignore-requires-python --requirement 'https://raw.githubusercontent.com/google-deepmind/concordia/main/examples/requirements.in' 'git+https://github.com/google-deepmind/concordia.git#egg=gdm-concordia'
  %pip list

In [ ]:
# @title Imports

import numpy as np
from IPython import display

import sentence_transformers

from concordia.language_model import utils as language_model_utils

from concordia.prefabs.simulation import generic as simulation

import concordia.prefabs.entity as entity_prefabs
import concordia.prefabs.game_master as game_master_prefabs

from concordia.typing import prefab as prefab_lib
from concordia.utils import helper_functions

In [9]:
# @title Language Model Selection: provide key or select DISABLE_LANGUAGE_MODEL

# By default this colab uses models via an external API so you must provide an
# API key. TogetherAI offers open weights models from all sources.

API_KEY = 'f36b08942cd3da2437d97b01c32f4edcf363c8ca594baa1254752c092dce7d5d'  #@param {type: 'string'}
# See concordia/language_model/utils.py
API_TYPE = 'together_ai'  # e.g. 'together_ai' or 'openai'.
MODEL_NAME = 'openai/gpt-oss-20b'  # for API_TYPE = 'together_ai', we recommend MODEL_NAME = 'google/gemma-3-27b-it'
# To debug without spending money on API calls, set DISABLE_LANGUAGE_MODEL=True
DISABLE_LANGUAGE_MODEL = False

In [10]:
# @title Use the selected language model

# Note that it is also possible to use local models or other API models,
# simply replace this cell with the correct initialization for the model
# you want to use.

if not DISABLE_LANGUAGE_MODEL and not API_KEY:
  raise ValueError('API_KEY is required.')

model = language_model_utils.language_model_setup(
    api_type=API_TYPE,
    model_name=MODEL_NAME,
    api_key=API_KEY,
    disable_language_model=DISABLE_LANGUAGE_MODEL,
)


In [11]:
# @title Setup sentence encoder

if DISABLE_LANGUAGE_MODEL:
  embedder = lambda _: np.ones(3)
else:
  st_model = sentence_transformers.SentenceTransformer(
      'sentence-transformers/all-mpnet-base-v2')
  embedder = lambda x: st_model.encode(x, show_progress_bar=False)

In [12]:
test = model.sample_text(
    'Is societal and technological progress like getting a clearer picture of '
    'something true and deep?')
print(test)

Societal and technological progress often feels like steadily sharpening a lens, bringing distant ideas into clearer focus. Every invention—whether a new communication medium, a medical breakthrough, or an environmental technology—expands the range of what we can perceive, measure, and influence. This progressive clarity unfolds in a few intertwined layers:

1. **Concrete Understanding of Complex Systems**  
   As knowledge accumulates, the intricate webs of cause and effect that once seemed opaque become more tractable. For example, the study of genetics, once a tangled maze of inheritance, now reveals precise pathways from DNA mutations to disease phenotypes. Each new algorithm or laboratory technique peels back another layer of that maze, allowing us to predict, manipulate, or intervene with increasing confidence.

2. **Democratization of Insight**  
   Advances in data collection and sharing mean that insights are no longer the exclusive domain of a few experts. Cloud computing, op

In [13]:
# @title Load prefabs from packages to make the specific palette to use here.

prefabs = {
    **helper_functions.get_package_classes(entity_prefabs),
    **helper_functions.get_package_classes(game_master_prefabs),
}

In [14]:
#@title Print menu of prefabs

display.display(
    display.Markdown(helper_functions.print_pretty_prefabs(prefabs)))

---
**`basic__Entity`**:
```python
Entity(
    description='An entity that makes decisions by asking "What situation am I in right now?", "What kind of person am I?", and "What would a person like me do in a situation like this?"',
    params={'name': 'Alice', 'goal': '', 'randomize_choices': True}
)
```
---
**`basic_scripted__Entity`**:
```python
Entity(
    description='An entity that makes decisions by asking "What situation am I in right now?", "What kind of person am I?", and "What would a person like me do in a situation like this?"',
    params={'name': 'Alice', 'goal': '', 'script': []}
)
```
---
**`basic_with_plan__Entity`**:
```python
Entity(
    description='An entity that makes decisions by asking "What situation am I in right now?", "What kind of person am I?", and "What would a person like me do in a situation like this?" and building a plan based on the answers. It then tries to execute the plan.',
    params={'name': 'Alice', 'goal': '', 'force_time_horizon': False}
)
```
---
**`conversational__Entity`**:
```python
Entity(
    description='An entity that participates in conversations, aiming to create a dynamically balanced and engaging dialogue.',
    params={'name': 'Debra'}
)
```
---
**`fake_assistant_with_configurable_system_prompt__Entity`**:
```python
Entity(
    description='An entity that simulates an AI assistant with a configurable system prompt.',
    params={'name': 'Assistant', 'system_prompt': 'Assistant is a helpful and harmless AI assistant.'}
)
```
---
**`minimal__Entity`**:
```python
Entity(
    description='An entity that has a minimal set of components and is configurable by the user. The initial set of components manage memory, observations, and instructions. If goal is specified, the entity will have a goal constant component.',
    params={'name': 'Alice', 'goal': '', 'custom_instructions': '', 'extra_components': {}, 'extra_components_index': {}, 'randomize_choices': True}
)
```
---
**`dialogic__GameMaster`**:
```python
GameMaster(
    description='A game master specialized for handling conversation.',
    params={'name': 'conversation rules', 'next_game_master_name': 'default rules', 'acting_order': 'game_master_choice', 'can_terminate_simulation': True}
)
```
---
**`dialogic_and_dramaturgic__GameMaster`**:
```python
GameMaster(
    description='A game master specialized for handling conversation. This game master is designed to be used with scenes.',
    params={'name': 'conversation rules', 'scenes': ()}
)
```
---
**`formative_memories_initializer__GameMaster`**:
```python
GameMaster(
    description='An initializer for all entities that generates formative memories from their childhood.',
    params={'name': 'initial setup rules', 'next_game_master_name': 'default rules', 'shared_memories': [], 'player_specific_context': {}, 'player_specific_memories': {}}
)
```
---
**`game_theoretic_and_dramaturgic__GameMaster`**:
```python
GameMaster(
    description='A game master specialized for handling matrix game. decisions, designed to be used with scenes.',
    params={'name': 'decision rules', 'scenes': (), 'action_to_scores': <function _default_action_to_scores at 0x7a24925eb880>, 'scores_to_observation': <function _default_scores_to_observation at 0x7a24925eb920>}
)
```
---
**`generic__GameMaster`**:
```python
GameMaster(
    description='A general purpose game master.',
    params={'name': 'default rules', 'extra_event_resolution_steps': '', 'extra_components': {}, 'extra_components_index': {}, 'acting_order': 'game_master_choice'}
)
```
---
**`interviewer__GameMaster`**:
```python
GameMaster(
    description='A game master that administers questionnaires to a specified player.',
    params={'name': 'InterviewerGM', 'player_names': [], 'questionnaires': [], 'verbose': False}
)
```
---
**`marketplace__GameMaster`**:
```python
GameMaster(
    description='A generic Game Master that administers a psychology experiment defined by custom observation and action specification components.',
    params={'name': 'ExperimenterGM', 'experiment_component_class': None, 'experiment_component_init_kwargs': {}}
)
```
---
**`open_ended_interviewer__GameMaster`**:
```python
GameMaster(
    description='A game master that administers questionnaires to a specified player.',
    params={'name': 'InterviewerGM', 'player_names': [], 'questionnaires': [], 'sequence_of_events': [], 'embedder': None, 'verbose': False}
)
```
---
**`psychology_experiment__GameMaster`**:
```python
GameMaster(
    description='A generic Game Master that administers a psychology experiment defined by custom observation and action specification components.',
    params={'name': 'ExperimenterGM', 'scenes': (), 'experiment_component_class': None, 'experiment_component_init_kwargs': {}}
)
```
---
**`scripted__GameMaster`**:
```python
GameMaster(
    description='A game master that administers questionnaires to a specified player.',
    params={'name': 'ScriptedGM', 'script': [], 'verbose': False}
)
```
---
**`situated__GameMaster`**:
```python
GameMaster(
    description='A general game master for games set in a specific location.',
    params={'name': 'default rules', 'extra_event_resolution_steps': '', 'locations': '', 'extra_components': {}, 'extra_components_index': {}}
)
```
---
**`situated_in_time_and_place__GameMaster`**:
```python
GameMaster(
    description='A general game master for games situated in a physical time/place.',
    params={'name': 'default rules', 'extra_event_resolution_steps': '', 'clock_description': "The passing of time can be conveyed using any convenient feature of the environment, e.g. a physical clock, the angle of the sun, extent of a candle's melting, phase of the moon, agricultural season, elapsed time since an event, etc. Whenever possible, try to track the day and year as well as the time within the day. To determine the passing of time, try to make reasonable inferences about the amount of time that would most likely have elapsed between the previous event and the latest event, taking into account the number of simulation steps taken.", 'start_time': '', 'locations': '', 'extra_components': {}, 'extra_components_index': {}}
)
```
---

In [15]:
# @title Configure instances.

instances = [
    prefab_lib.InstanceConfig(
        prefab='basic__Entity',
        role=prefab_lib.Role.ENTITY,
        params={
            'name': 'Oliver Cromwell',
            'goal': 'become lord protector',
        },
    ),
    prefab_lib.InstanceConfig(
        prefab='basic__Entity',
        role=prefab_lib.Role.ENTITY,
        params={
            'name': 'King Charles I',
            'goal': 'avoid execution for treason',
        },
    ),
    prefab_lib.InstanceConfig(
        prefab='generic__GameMaster',
        role=prefab_lib.Role.GAME_MASTER,
        params={
            'name': 'default rules',
            # Comma-separated list of thought chain steps.
            'extra_event_resolution_steps': '',
        },
    ),
    prefab_lib.InstanceConfig(
        prefab='formative_memories_initializer__GameMaster',
        role=prefab_lib.Role.INITIALIZER,
        params={
            'name': 'initial setup rules',
            'next_game_master_name': 'default rules',
            'shared_memories': [
                'The king was captured by Parliamentary forces in 1646.',
                'Charles I was tried for treason and found guilty.',
            ],
        },
    ),
]

In [16]:
config = prefab_lib.Config(
    default_premise='Today is January 29, 1649.',
    default_max_steps=5,
    prefabs=prefabs,
    instances=instances,
)

# The simulation

In [17]:
# @title Initialize the simulation
runnable_simulation = simulation.Simulation(
    config=config,
    model=model,
    embedder=embedder,
)

In [18]:
# @title Run the simulation
raw_log = []
results_log = runnable_simulation.play(max_steps=5,
                                       raw_log=raw_log)

Terminate? No
Game master: initial setup rules
Entity Oliver Cromwell observed: The king was captured by Parliamentary forces in 1646.


Charles I was tried for treason and found guilty.


When Oliver Cromwell was five years old, he watched his father repair a broken mill wheel and realized that even a simple task required patience and skill. He offered to fetch water from the river, and his father praised him for taking initiative, planting the seed of responsibility. That night, the family gathered around a candle, and his mother sang hymns that echoed the hope of a better future, imprinting faith in Oliver’s heart. He slept with a sense of purpose, dreaming of the day he could serve his community. This early memory taught Oliver that duty begins with small actions.

When Oliver Cromwell was twenty, he marched with the Parliament's militia into a town threatened by royalist forces. Amidst the clash, he shielded a wounded child, earning gratitude from the townsfolk and a reputation fo

In [19]:
# @title Display the log
display.HTML(results_log)

```
Copyright 2024 DeepMind Technologies Limited.

Licensed under the Apache License, Version 2.0 (the "License");
you may not use this file except in compliance with the License.
You may obtain a copy of the License at

    https://www.apache.org/licenses/LICENSE-2.0

Unless required by applicable law or agreed to in writing, software
distributed under the License is distributed on an "AS IS" BASIS,
WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
See the License for the specific language governing permissions and
limitations under the License.
```